# Tomahawk 16 T MGRID + 22 GWₑ screen — run 1 of 3

Attach **Tomahawk_G1_G2_All_Inputs.zip** as one Kaggle dataset using **Add Input** (the ZIP may appear as an extracted folder). Then click **Run All**. This run checks planes 0–31 of the existing 96-plane grid and runs its power scenarios. Download the small results ZIP from the Output pane. Each notebook can run independently with the same input.

The power screen tests assumptions; a 16 T vacuum MGRID does not determine fusion power.


In [ ]:
RUN_INDEX = 1
assert RUN_INDEX in (1, 2, 3)


In [ ]:
import importlib
import os
import subprocess
import sys
from pathlib import Path

# Kaggle's Python image normally has both. Install only if its image is missing one.
for package in ("numpy", "scipy"):
    try:
        importlib.import_module(package)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

import numpy as np
import scipy
from scipy.io import netcdf_file
print("Python", sys.version.split()[0], "NumPy", np.__version__, "SciPy", scipy.__version__)


In [ ]:
import contextlib
import csv
import hashlib
import io
import json
import lzma
import zipfile

MANIFEST = "MANIFEST_SHA256.json"
TF_CSV = "Tomahawk_TF48_16FP_16T_CAD_Handoff/TF48_exact_centerlines_m.csv"
MGRID_XZ = "Tomahawk_G2_16T_Preflight/mgrid_tomahawk_16T_TF_PF_16FP.nc.xz"
EXPECTED_MGRID_SHA256 = "b5758ac6235593d4f80724078cecfcca1e23127364bf1c76aa7d2a8b5d925a2b"

class InputPackage:
    def __init__(self, path, kind, prefix):
        self.path, self.kind, self.prefix = Path(path), kind, prefix
        self.archive = zipfile.ZipFile(path) if kind == "zip" else None

    def open(self, relative_path):
        if self.archive:
            return self.archive.open(self.prefix + relative_path)
        return (self.path / relative_path).open("rb")

    def close(self):
        if self.archive:
            self.archive.close()

def locate_input():
    roots = []
    if os.environ.get("TOMAHAWK_INPUT_PATH"):
        roots.append(Path(os.environ["TOMAHAWK_INPUT_PATH"]))
    roots += [Path("/kaggle/input"), Path.cwd()]
    seen = set()
    for root in roots:
        if not root.exists():
            continue
        candidates = [root]
        if root.is_dir():
            candidates += [p.parent for p in sorted(root.rglob(MANIFEST))]
            candidates += sorted(root.rglob("*.zip"))
        for candidate in candidates:
            if candidate in seen:
                continue
            seen.add(candidate)
            if (candidate.is_dir() and (candidate / MANIFEST).is_file()
                    and (candidate / TF_CSV).is_file()
                    and (candidate / MGRID_XZ).is_file()):
                return InputPackage(candidate, "directory", "")
            if candidate.is_file() and candidate.suffix.lower() == ".zip":
                try:
                    with zipfile.ZipFile(candidate) as z:
                        names = z.namelist()
                        for name in names:
                            if name.endswith("/" + MANIFEST) or name == MANIFEST:
                                prefix = name[:-len(MANIFEST)]
                                if prefix + TF_CSV in names and prefix + MGRID_XZ in names:
                                    return InputPackage(candidate, "zip", prefix)
                except (zipfile.BadZipFile, OSError):
                    continue
    raise FileNotFoundError(
        "Attach Tomahawk_G1_G2_All_Inputs.zip with Kaggle Add Input. "
        "This notebook accepts either the ZIP or its extracted folder under /kaggle/input."
    )

package = locate_input()
print("Input:", package.path, "mode:", package.kind)
with package.open(MANIFEST) as stream:
    manifest = json.load(stream)
assert isinstance(manifest, dict) and TF_CSV in manifest and MGRID_XZ in manifest
for relative, expected in manifest.items():
    digest = hashlib.sha256()
    with package.open(relative) as stream:
        for block in iter(lambda: stream.read(4 * 1024 * 1024), b""):
            digest.update(block)
    if digest.hexdigest() != expected:
        raise ValueError("SHA-256 mismatch: " + relative)
print("Input manifest verified:", len(manifest), "files")

with package.open(TF_CSV) as stream:
    tf_rows = list(csv.DictReader(io.TextIOWrapper(stream, encoding="utf-8-sig")))
coil_ids = {row["coil_id"] for row in tf_rows}
assert len(tf_rows) == 48 * 512 and len(coil_ids) == 48
assert all(int(row["turns"]) == 100 for row in tf_rows)
assert all({int(row["segment"]) for row in tf_rows if row["coil_id"] == c} == set(range(512)) for c in coil_ids)
first = np.array([[float(row[key]) for key in ("x_m", "y_m", "z_m")] for row in tf_rows])
last = np.array([[float(row[key]) for key in ("next_x_m", "next_y_m", "next_z_m")] for row in tf_rows])
segment_vectors = last - first
segment_mids = (first + last) / 2
assert np.isfinite(first).all() and np.isfinite(last).all()
print("TF centerlines: 48 coils x 512 segments, 100 turns each")


In [ ]:
import time
working_root = Path(os.environ.get("TOMAHAWK_WORKING_DIR", "/kaggle/working" if Path("/kaggle/working").exists() else str(Path.cwd() / "local_kaggle_working")))
working_root.mkdir(parents=True, exist_ok=True)
nc_path = working_root / ("tomahawk_mgrid_run_%d.nc" % RUN_INDEX)

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

if not nc_path.exists() or sha256_file(nc_path) != EXPECTED_MGRID_SHA256:
    nc_path.unlink(missing_ok=True)
    tmp_path = nc_path.with_suffix(".partial")
    digest = hashlib.sha256()
    try:
        with package.open(MGRID_XZ) as source, lzma.LZMAFile(source, "rb") as decoded, tmp_path.open("wb") as target:
            for block in iter(lambda: decoded.read(8 * 1024 * 1024), b""):
                digest.update(block)
                target.write(block)
        if digest.hexdigest() != EXPECTED_MGRID_SHA256:
            raise ValueError("Decompressed MGRID SHA-256 mismatch")
        tmp_path.replace(nc_path)
    finally:
        tmp_path.unlink(missing_ok=True)
print("Verified 16 T MGRID:", nc_path.stat().st_size, "bytes, SHA-256", EXPECTED_MGRID_SHA256)


In [ ]:
import math

plane_start = (RUN_INDEX - 1) * 32
plane_stop = RUN_INDEX * 32
np.seterr(invalid="raise", divide="raise", over="raise")

def direct_tf_cylindrical(plane, radial_index, vertical_index):
    angle = 2 * math.pi * plane / (16 * 96)
    radius = 4.0 + .035 * radial_index
    z = -3.5 + .035 * vertical_index
    sample = np.array([radius * math.cos(angle), radius * math.sin(angle), z])
    offset = sample - segment_mids
    distance = np.linalg.norm(offset, axis=1)
    # Midpoint Biot-Savart quadrature: 100 turns x -125 kA per turn, mu_0/(4*pi)=1e-7.
    cart = 1e-7 * (-125000.0 * 100) * np.sum(
        np.cross(segment_vectors, offset) / distance[:, None] ** 3, axis=0)
    return [float(cart[0] * math.cos(angle) + cart[1] * math.sin(angle)),
            float(-cart[0] * math.sin(angle) + cart[1] * math.cos(angle)),
            float(cart[2])]

plane_rows, direct_rows = [], []
with netcdf_file(str(nc_path), mode="r", mmap=True) as grid:
    expected_dims = {"rad": 201, "zee": 201, "phi": 96}
    assert all(grid.dimensions[k] == v for k, v in expected_dims.items())
    for k, value in (("ir",201),("jz",201),("kp",96),("nfp",16),("nextcur",7)):
        assert int(grid.variables[k].getValue()) == value, (k, value)
    assert float(grid.variables["rmin"].getValue()) == 4.0
    assert float(grid.variables["rmax"].getValue()) == 11.0
    assert float(grid.variables["zmin"].getValue()) == -3.5
    assert float(grid.variables["zmax"].getValue()) == 3.5
    currents = np.array(grid.variables["raw_coil_cur"][:], dtype=np.float64).copy()
    assert currents.shape == (7,) and float(currents[0]) == -125000.0
    names = [bytes(grid.variables["coil_group"][i]).decode("ascii").strip() for i in range(7)]
    assert names == ["TF48", "PF_L_IN", "PF_L_OUT", "PF_M_IN", "PF_M_OUT", "PF_U_IN", "PF_U_OUT"]
    for group in range(1, 8):
        for component in ("br", "bp", "bz"):
            assert grid.variables[f"{component}_{group:03d}"].shape == (96, 201, 201)

    selected_planes = {plane_start, plane_start + 8, plane_start + 16,
                       plane_start + 24, plane_stop - 1}
    sample_nodes = [(100,100), (80,120), (120,80)]  # (R index, Z index)
    for plane in range(plane_start, plane_stop):
        combined = [np.zeros((201, 201), dtype=np.float64) for _ in range(3)]
        for group_index, current in enumerate(currents, start=1):
            for index, component in enumerate(("br", "bp", "bz")):
                cell = np.asarray(grid.variables[f"{component}_{group_index:03d}"][plane], dtype=np.float64)
                if not np.isfinite(cell).all():
                    raise ValueError(f"Non-finite MGRID value at plane {plane}, {component}, group {group_index}")
                combined[index] += float(current) * cell
        core = tuple(arr[65:136, 65:136] for arr in combined)
        magnitude = np.sqrt(core[0]**2 + core[1]**2 + core[2]**2)
        plane_rows.append({"plane": plane, "angle_rad": 2*math.pi*plane/(16*96),
                           "axis_Bphi_total_T": float(combined[1][100,100]),
                           "axis_Bmag_total_T": float(np.linalg.norm([arr[100,100] for arr in combined])),
                           "core_Bmag_min_T": float(magnitude.min()),
                           "core_Bmag_max_T": float(magnitude.max())})
        if plane in selected_planes:
            for radial_index, vertical_index in sample_nodes:
                mgrid_tf = [float(currents[0] * grid.variables[f"{component}_001"][plane, vertical_index, radial_index])
                            for component in ("br", "bp", "bz")]
                direct_tf = direct_tf_cylindrical(plane, radial_index, vertical_index)
                relative = float(np.linalg.norm(np.array(direct_tf) - mgrid_tf) / np.linalg.norm(mgrid_tf))
                direct_rows.append({"plane":plane, "radial_index":radial_index,
                                    "vertical_index":vertical_index,
                                    "TF_mgrid_T":mgrid_tf, "TF_direct_T":direct_tf,
                                    "relative_vector_error":relative})

assert len(plane_rows) == 32 and len(direct_rows) == 15
max_relative_error = max(row["relative_vector_error"] for row in direct_rows)
assert max_relative_error < 0.0005, f"TF centerline mismatch: {max_relative_error:.6g}"
assert all(math.isfinite(row["axis_Bmag_total_T"]) for row in plane_rows)
print(f"Run {RUN_INDEX}: processed all 7 current groups over planes {plane_start}..{plane_stop-1}")
print("Max TF centerline-vs-grid relative vector error:", f"{max_relative_error:.6g}")
print("Axis combined |B| first plane:", f"{plane_rows[0]['axis_Bmag_total_T']:.8f} T")


In [ ]:
# Accounting screen only: no 16 T field-to-fusion-source assumption.
USER_ANCHOR_GWE = 15.08
TARGET_GWE = 22.0
ACTIVE = 10
SOURCE_BASE = 5.0
N_CAPTURE_BASE = .95
A_DEPOSITION_BASE = .90
ETA_BASE = .40
RECIRC_BASE_GWE = 3.0

def usable_fraction(ncapture, alpha_dep):
    return 14.1 / 17.6 * ncapture + 3.5 / 17.6 * alpha_dep

def net_gwe(active, source, eta, recirc, ncapture, alpha_dep):
    return active * source * usable_fraction(ncapture, alpha_dep) * eta - recirc

baseline_model = net_gwe(ACTIVE, SOURCE_BASE, ETA_BASE, RECIRC_BASE_GWE,
                         N_CAPTURE_BASE, A_DEPOSITION_BASE)
unattributed_delta = baseline_model - USER_ANCHOR_GWE
assert abs(baseline_model - 15.801136363636363) < 1e-10

def screen(name, source, eta, recirc, active=ACTIVE,
           ncapture=N_CAPTURE_BASE, alpha_dep=A_DEPOSITION_BASE):
    model = net_gwe(active, source, eta, recirc, ncapture, alpha_dep)
    anchored = model - unattributed_delta
    return {"scenario":name, "active_modules":active,
            "source_GW_fusion_per_module":source, "conversion_fraction":eta,
            "total_recirc_GWe":recirc, "neutron_capture_fraction":ncapture,
            "alpha_deposition_fraction":alpha_dep,
            "net_model_GWe":round(model, 9),
            "net_with_15p08_offset_GWe":round(anchored, 9),
            "extra_above_15p08_GWe":round(anchored - USER_ANCHOR_GWE, 9),
            "reaches_22_with_offset":anchored >= TARGET_GWE - 1e-12}

power_rows = [screen("documented_model_baseline",5.0,.40,3.0)]
if RUN_INDEX == 1:
    # Source requirement at fixed 40% conversion and 3 GW total recirculation.
    power_rows += [screen("source_sweep",round(5.0 + .2*i,8),.40,3.0) for i in range(13)]
elif RUN_INDEX == 2:
    # Conversion and electrical load at the original assumed 5 GW/core source.
    power_rows += [screen("conversion_recirc_sweep",5.0,round(.40 + .02*i,8),recirc)
                   for i in range(11) for recirc in (2.5,3.0,3.5,4.0)]
else:
    # Combined source/efficiency/recirculation and poorer energy-capture cases.
    power_rows += [screen("combined_capture_sweep",round(5.0 + .25*i,8),
                          round(.40 + .02*j,8),recirc,ncapture=ncap,alpha_dep=adep)
                   for i in range(9) for j in range(6) for recirc in (3.0,3.5)
                   for ncap,adep in ((.95,.90),(.90,.80))]
    power_rows.append(screen("explicit_6p1_46pct_3p5",6.1,.46,3.5))

required_source_core = (TARGET_GWE + RECIRC_BASE_GWE + unattributed_delta) / (
    ACTIVE * ETA_BASE * usable_fraction(N_CAPTURE_BASE, A_DEPOSITION_BASE))
required_eta_5gw = (TARGET_GWE + RECIRC_BASE_GWE + unattributed_delta) / (
    ACTIVE * SOURCE_BASE * usable_fraction(N_CAPTURE_BASE, A_DEPOSITION_BASE))
print(f"Power screen {RUN_INDEX}: {len(power_rows)} scenarios, documented model {baseline_model:.6f} GWe")
print(f"15.08 anchor retains unexplained {unattributed_delta:.6f} GW offset")
print(f"22 GWe requires {required_source_core:.4f} GW fusion/core at 40%, 3 GW recirc (anchored)")
print(f"or {required_eta_5gw*100:.2f}% conversion at 5 GW/core, 3 GW recirc (anchored)")
print("Scenarios reaching 22 with offset:",sum(r["reaches_22_with_offset"] for r in power_rows))


In [ ]:
import datetime

report = {
    "run":RUN_INDEX, "completed_utc":datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "input_path":str(package.path), "source_mgrid_sha256":EXPECTED_MGRID_SHA256,
    "manifest_file_count":len(manifest), "plane_range_inclusive":[plane_start,plane_stop-1],
    "mgrid_shape":[96,201,201], "field_periods":16,
    "current_groups":names, "raw_coil_currents_A":currents.tolist(),
    "finite_coverage":"all 201x201 nodes, all 7 groups and all 3 components in these 32 planes",
    "direct_tf_samples":15, "max_tf_relative_vector_error":max_relative_error,
    "field_planes":plane_rows, "tf_direct_checks":direct_rows,
    "power_basis":{"user_anchor_GWe":USER_ANCHOR_GWE,"target_GWe":TARGET_GWE,
                   "documented_assumption_baseline_GWe":baseline_model,
                   "unattributed_model_minus_user_anchor_GW":unattributed_delta,
                   "required_source_GW_per_module_40pct_3GW_anchored":required_source_core,
                   "required_conversion_fraction_5GW_3GW_anchored":required_eta_5gw,
                   "laser_on_electric_GWe_10x25MW_optical_15pct_wallplug":10*.025/.15},
    "power_scenarios":power_rows,
    "interpretation":"Vacuum TF/PF MGRID audit and algebraic power scenarios. "
                     "A 16 T vacuum field is not a D-T fusion-power prediction. "
                     "The source, capture, conversion, and recirculation changes remain unverified. "
                     "The 0.721 GW baseline offset has no identified physical cause."
}
json_path = working_root / f"Tomahawk_Run_{RUN_INDEX}_16T_22GWe_results.json"
json_path.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")

csv_path = working_root / f"Tomahawk_Run_{RUN_INDEX}_power_scenarios.csv"
with csv_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(power_rows[0]))
    writer.writeheader()
    writer.writerows(power_rows)

readme_path = working_root / f"Tomahawk_Run_{RUN_INDEX}_READ_ME.txt"
readme_path.write_text(
    f"Tomahawk Kaggle run {RUN_INDEX} of 3\n"
    f"MGRID planes processed: {plane_start}..{plane_stop-1} of 96. "
    "All seven groups and all 201 x 201 nodes scanned per plane.\n"
    f"TF independent samples: 15; maximum relative vector error: {max_relative_error:.7g}.\n"
    "Power cases are arithmetic requirements and do not establish a 16 T fusion source.\n"
    "See JSON and CSV. All three notebooks independently consume the same input ZIP.\n",
    encoding="utf-8")

results_zip = working_root / f"Tomahawk_Run_{RUN_INDEX}_16T_22GWe_results.zip"
with zipfile.ZipFile(results_zip, "w", compression=zipfile.ZIP_DEFLATED) as result_archive:
    for path in (json_path, csv_path, readme_path):
        result_archive.write(path, arcname=path.name)

package.close()
nc_path.unlink(missing_ok=True)  # Do not export a duplicate 650 MB grid from Kaggle.
print("DONE:", results_zip)
print("Download this results ZIP from the Kaggle Output pane.")
